# CGC Configuration + Parameter Monitor

Activate a configuration on all 4 PSUs and both switches (normal `swB` + high resolution `swA`),
run a lightweight housekeeping loop, and a web-based live plot of temperatures + PSU voltages +
PSU currents. Power dissipation (switch + PSU regulator) is computed and logged every cycle.

## Hardware mapping

| Device | Channels | PSU | Load                    |
|--------|----------|-----|-------------------------|
| swB    | out 0/1  | psu2| Quadrupole 1            |
| swB    | out 2/3  | psu1| Ion Funnels             |
| swA    | out 0/1  | psu3| Quadrupole 2            |
| swA    | out 2/3  | psu4| Quadrupoles 3/4         |

## Safety limits (email from CGC)

- Per-switch-channel dissipation must stay under **100 W** (switch pair ≤ 200 W).
- At 350 V that corresponds to ~285 mA; **limit current to ≤ 300 mA** for high-frequency tests.
- Switch power  = `V_out · I_out` (both from `get_psu_data`).
- PSU regulator = `V_dropout · I_out`.
- Recipe: first ramp **voltage** at low freq (1 kHz) to 350 V, then ramp **frequency** at full voltage.

## Imports, Logger, Device Instances

In [ ]:
import sys
import os
import time
import logging
from datetime import datetime
from pathlib import Path

# Add src to path
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'src'))

from devices.cgc.psu.psu import PSU
from devices.cgc.sw.sw import SW
from devices.cgc.sw_HR.sw_HR import SWHR

In [ ]:
# Setup external logger
repo_root = Path(os.getcwd()).parent.parent
log_dir = repo_root / "debugging" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = log_dir / f"023_cgc_config_monitor_{timestamp}.log"

logger = logging.getLogger(f"023_cgc_config_monitor_{timestamp}")
logger.setLevel(logging.DEBUG)

file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(console_handler)

logger.info("Logger initialized.")
print(f"Log file: {log_file}")

In [ ]:
psu1 = PSU("psu1", com=3, port=0, logger=logger)
psu2 = PSU("psu2", com=4, port=1, logger=logger)
psu3 = PSU("psu3", com=5, port=2, logger=logger)
psu4 = PSU("psu4", com=6, port=3, logger=logger)
swA = SWHR("swA", com=10, stream=0, logger=logger)  # high resolution switch
swB = SW("swB", com=13, port=0, logger=logger)      # normal switch

## Connect All Devices

In [ ]:
psu1.connect()
psu1.set_comspeed(230400)

psu2.connect()
psu2.set_comspeed(230400)

psu3.connect()
psu3.set_comspeed(230400)

psu4.connect()
psu4.set_comspeed(230400)

swA.connect()
swA.set_comspeed(230400)

swB.connect()
swB.set_comspeed(230400)

## Connection Check (read temperature from each device)

In [ ]:
for name, dev in [("psu1", psu1), ("psu2", psu2), ("psu3", psu3), ("psu4", psu4),
                  ("swA", swA), ("swB", swB)]:
    print(f"{name}: {dev.get_sensor_data()}")

## Apply Configuration

Switches first, then PSUs. Set the desired configuration numbers below.

In [ ]:
# --- Switches first ---
swA_config = 0  # TODO: set high-resolution switch config number
swB_config = 0  # TODO: set normal switch config number

# Acquire each device's thread_lock so this is safe to run while the live plot is polling.
with swA.thread_lock:
    swA.load_current_config(swA_config)
with swB.thread_lock:
    swB.load_current_config(swB_config)

In [ ]:
# --- Then PSUs ---
psu1_config = 0  # TODO: set PSU1 config number
psu2_config = 0  # TODO: set PSU2 config number
psu3_config = 0  # TODO: set PSU3 config number
psu4_config = 0  # TODO: set PSU4 config number

# Acquire each device's thread_lock so this is safe to run while the live plot is polling.
with psu1.thread_lock:
    psu1.load_current_config(psu1_config)
with psu2.thread_lock:
    psu2.load_current_config(psu2_config)
with psu3.thread_lock:
    psu3.load_current_config(psu3_config)
with psu4.thread_lock:
    psu4.load_current_config(psu4_config)

## Simplified Housekeeping Loop

Reads temperature (`get_sensor_data`) and device state (`get_device_state`) from every device every 10s.
Stop with KeyboardInterrupt.

In [ ]:
interval = 10  # seconds

devices = [
    ("psu1", psu1),
    ("psu2", psu2),
    ("psu3", psu3),
    ("psu4", psu4),
    ("swA", swA),
    ("swB", swB),
]

def hk_cycle():
    for name, dev in devices:
        try:
            with dev.thread_lock:
                s_status, t0, t1, t2 = dev.get_sensor_data()
                d_status, state_hex, state_names = dev.get_device_state()
            logger.info(
                f"{name} | T0={t0:.1f} T1={t1:.1f} T2={t2:.1f} degC | "
                f"state={state_hex} {state_names}"
            )
        except Exception as e:
            logger.error(f"{name} housekeeping failed: {e}")

logger.info(f"Starting housekeeping loop (interval={interval}s). Ctrl+C to stop.")
try:
    while True:
        hk_cycle()
        time.sleep(interval)
except KeyboardInterrupt:
    logger.info("Housekeeping loop stopped.")

## Live Monitor (web-based, opens in Chrome)

Bokeh server at ~1 Hz. Layout: two columns.

**Left column** — 6 temperature subplots (one per device, broken sensors hidden).

**Right column** — per PSU: voltage (V_pos, V_neg) and current (I_pos, I_neg, with 300 mA dashed limit line).

**Logged every cycle** (file only) — per PSU channel: `V, I, V_dropout, P_switch = V*I, P_psu = V_dropout*I`.
At high RF frequency these are DC averages — exactly the thermally relevant values for the 100 W/channel limit.

Stop with the cell below.

In [ ]:
import threading
import webbrowser
from datetime import timedelta

from bokeh.plotting import figure
from bokeh.layouts import column, row
from bokeh.models import ColumnDataSource, DatetimeTickFormatter, Range1d, Span
from bokeh.palettes import Category10
from bokeh.server.server import Server
from tornado.ioloop import IOLoop

# Plot settings
POLL_INTERVAL_MS = 1000   # 1 Hz update rate
ROLLING_WINDOW_MIN = 5    # minutes of data visible in plot
MAX_POINTS = 7200         # ~2 hours of data at 1 Hz
BOKEH_PORT = 5007         # different from 020's 5006 in case both run

CURRENT_LIMIT_A = 0.300   # 300 mA safety line (email: ~285 mA at 350 V, 650 kHz)

# Hardware mapping for titles
PSU_LABELS = {
    "psu1": "PSU1 -> IonFunnels (swB out 2/3)",
    "psu2": "PSU2 -> Quad1 (swB out 0/1)",
    "psu3": "PSU3 -> Quad2 (swA out 0/1)",
    "psu4": "PSU4 -> Quad3/4 (swA out 2/3)",
}
SW_LABELS = {
    "swA": "swA (high-res) -> Quad2/3/4",
    "swB": "swB (normal)  -> Quad1 + IonFunnels",
}

TEMP_DEVICES = [
    ("psu1", psu1),
    ("psu2", psu2),
    ("psu3", psu3),
    ("psu4", psu4),
    ("swA",  swA),
    ("swB",  swB),
]
PSU_DEVICES = [
    ("psu1", psu1),
    ("psu2", psu2),
    ("psu3", psu3),
    ("psu4", psu4),
]

# Which sensor indices to plot for each device (broken sensors hidden from plot only).
# swA: sensor 2 broken -> plot 0,1
# swB: sensor 0 broken -> plot 1,2
PLOT_SENSORS = {
    "psu1": (0, 1, 2),
    "psu2": (0, 1, 2),
    "psu3": (0, 1, 2),
    "psu4": (0, 1, 2),
    "swA":  (0, 1),
    "swB":  (1, 2),
}

# Suppress console logging during live monitoring (file logging continues)
console_handler.setLevel(logging.WARNING)


def make_document(doc):
    """Build the Bokeh document: temps (left), PSU V/I (right)."""
    # --- data sources ---
    temp_sources = {}
    for name, _ in TEMP_DEVICES:
        cols = {"time": []}
        for s in PLOT_SENSORS[name]:
            cols[f"t{s}"] = []
        temp_sources[name] = ColumnDataSource(data=cols)

    psu_sources = {}
    for name, _ in PSU_DEVICES:
        psu_sources[name] = ColumnDataSource(data={
            "time": [], "v0": [], "v1": [], "i0": [], "i1": [],
        })

    now = datetime.now()
    x_range = Range1d(start=now - timedelta(minutes=ROLLING_WINDOW_MIN), end=now)

    # --- left column: temperatures ---
    colors = Category10[3]
    temp_figs = []
    for i, (name, _) in enumerate(TEMP_DEVICES):
        title = SW_LABELS[name] if name in SW_LABELS else PSU_LABELS[name]
        p = figure(
            title=f"Temp: {title}",
            x_axis_type="datetime",
            height=170, width=900,
            x_range=x_range,
            tools="pan,wheel_zoom,box_zoom,reset,save",
            active_scroll="wheel_zoom",
        )
        for s in PLOT_SENSORS[name]:
            p.line("time", f"t{s}", source=temp_sources[name],
                   line_width=2, color=colors[s], legend_label=f"Sensor {s}")
        p.yaxis.axis_label = "Temp [degC]"
        p.xaxis.formatter = DatetimeTickFormatter(
            seconds="%H:%M:%S", minsec="%H:%M:%S", minutes="%H:%M:%S",
            hourmin="%H:%M:%S", hours="%H:%M:%S",
        )
        p.legend.location = "top_left"
        p.legend.click_policy = "hide"
        p.xaxis.axis_label = "Time" if i == len(TEMP_DEVICES) - 1 else ""
        temp_figs.append(p)

    # --- right column: per-PSU voltage + current ---
    rf_figs = []
    for j, (name, _) in enumerate(PSU_DEVICES):
        # Voltage
        pv = figure(
            title=f"V: {PSU_LABELS[name]}",
            x_axis_type="datetime",
            height=140, width=900,
            x_range=x_range,
            tools="pan,wheel_zoom,box_zoom,reset,save",
            active_scroll="wheel_zoom",
        )
        pv.line("time", "v0", source=psu_sources[name],
                line_width=2, color=colors[0], legend_label="V_pos (ch0)")
        pv.line("time", "v1", source=psu_sources[name],
                line_width=2, color=colors[1], legend_label="V_neg (ch1)")
        pv.yaxis.axis_label = "V [V]"
        pv.xaxis.formatter = DatetimeTickFormatter(
            seconds="%H:%M:%S", minsec="%H:%M:%S", minutes="%H:%M:%S",
            hourmin="%H:%M:%S", hours="%H:%M:%S",
        )
        pv.legend.location = "top_left"
        pv.legend.click_policy = "hide"
        pv.xaxis.axis_label = ""
        rf_figs.append(pv)

        # Current
        pi = figure(
            title=f"I: {PSU_LABELS[name]}",
            x_axis_type="datetime",
            height=160, width=900,
            x_range=x_range,
            tools="pan,wheel_zoom,box_zoom,reset,save",
            active_scroll="wheel_zoom",
        )
        pi.line("time", "i0", source=psu_sources[name],
                line_width=2, color=colors[0], legend_label="I_pos (ch0)")
        pi.line("time", "i1", source=psu_sources[name],
                line_width=2, color=colors[1], legend_label="I_neg (ch1)")
        # Dashed 300 mA limit line
        limit = Span(location=CURRENT_LIMIT_A, dimension="width",
                     line_color="red", line_dash="dashed", line_width=1.5)
        pi.add_layout(limit)
        pi.yaxis.axis_label = "I [A]"
        pi.xaxis.formatter = DatetimeTickFormatter(
            seconds="%H:%M:%S", minsec="%H:%M:%S", minutes="%H:%M:%S",
            hourmin="%H:%M:%S", hours="%H:%M:%S",
        )
        pi.legend.location = "top_left"
        pi.legend.click_policy = "hide"
        pi.xaxis.axis_label = "Time" if j == len(PSU_DEVICES) - 1 else ""
        rf_figs.append(pi)

    layout = row(
        column(*temp_figs, sizing_mode="fixed"),
        column(*rf_figs,   sizing_mode="fixed"),
        sizing_mode="stretch_width",
    )
    doc.add_root(layout)
    doc.title = "CGC Configuration + Parameter Monitor"

    def update():
        try:
            now = datetime.now()

            # --- temps + device state/enable for every device ---
            for name, dev in TEMP_DEVICES:
                try:
                    with dev.thread_lock:
                        s_status, t0, t1, t2 = dev.get_sensor_data()
                        d_status, state_hex, state_names = dev.get_device_state()
                        e_status, enabled = dev.get_device_enable()

                    if s_status == dev.NO_ERR:
                        sensor_vals = (t0, t1, t2)
                        new_data = {"time": [now]}
                        for s in PLOT_SENSORS[name]:
                            new_data[f"t{s}"] = [sensor_vals[s]]
                        temp_sources[name].stream(new_data, rollover=MAX_POINTS)

                        logger.info(f"{name} | T0={t0:.1f} T1={t1:.1f} T2={t2:.1f} degC")
                        logger.info(f"{name} | state={state_hex} {state_names}")
                        logger.info(f"{name} | enabled={enabled}")
                except Exception as e:
                    logger.error(f"{name} sensor read failed: {e}")

            # --- PSU V / I / V_dropout / power ---
            for name, dev in PSU_DEVICES:
                try:
                    with dev.thread_lock:
                        s0, v0, i0, vd0 = dev.get_psu0_data()
                        s1, v1, i1, vd1 = dev.get_psu1_data()

                    if s0 == dev.NO_ERR and s1 == dev.NO_ERR:
                        psu_sources[name].stream({
                            "time": [now],
                            "v0": [v0], "v1": [v1],
                            "i0": [i0], "i1": [i1],
                        }, rollover=MAX_POINTS)

                        p_sw0  = v0 * i0
                        p_sw1  = v1 * i1
                        p_psu0 = vd0 * i0
                        p_psu1 = vd1 * i1
                        logger.info(
                            f"{name} | ch0 V={v0:.2f}V I={i0*1000:.1f}mA "
                            f"Vdrop={vd0:.2f}V P_sw={p_sw0:.1f}W P_psu={p_psu0:.1f}W"
                        )
                        logger.info(
                            f"{name} | ch1 V={v1:.2f}V I={i1*1000:.1f}mA "
                            f"Vdrop={vd1:.2f}V P_sw={p_sw1:.1f}W P_psu={p_psu1:.1f}W"
                        )
                except Exception as e:
                    logger.error(f"{name} PSU read failed: {e}")

            x_range.start = now - timedelta(minutes=ROLLING_WINDOW_MIN)
            x_range.end = now
        except Exception as e:
            logger.error(f"Plot update failed: {e}")

    doc.add_periodic_callback(update, POLL_INTERVAL_MS)


def _start_server():
    io_loop = IOLoop()
    io_loop.make_current()
    server = Server(
        {"/": make_document},
        port=BOKEH_PORT,
        io_loop=io_loop,
        allow_websocket_origin=[f"localhost:{BOKEH_PORT}"],
    )
    server.start()
    io_loop.start()


server_thread = threading.Thread(target=_start_server, daemon=True)
server_thread.start()

time.sleep(1)  # give the server a moment to start
url = f"http://localhost:{BOKEH_PORT}/"
try:
    chrome = webbrowser.get("C:/Program Files/Google/Chrome/Application/chrome.exe %s")
    chrome.open(url)
    print(f"Opened Chrome at {url}")
except webbrowser.Error:
    webbrowser.open(url)
    print(f"Chrome not found, opened default browser at {url}")

print(f"Bokeh server running at {url}")
print("Server runs in background. Run the next cell to restore console logging.")

# Test Workflow — change configs while the live monitor runs

Recipe from the CGC email:

1. **Low frequency, voltage ramp.** Set switch to 1 kHz SwitchSym (`swB` config 40).
   Ramp PSU voltage from ~10 V upwards while watching the current on the live plot.
2. **Full voltage, frequency ramp.** At full voltage, step switch frequency up
   (10 kHz → 100 kHz → 250 kHz → ...). Current scales linearly with frequency.

Each cell below edits **one** device. Load PSU configs only *after* the corresponding
switch has been configured and is running. All calls take the device `thread_lock`,
so they are safe to run concurrently with the live plot.

## swB test — PSU2 (Quad1) + PSU1 (IonFunnels)

**swB SwitchSym configs:** `1`=Standby, `40`=1 kHz, `50`=10 kHz, `60`=100 kHz,
`70`=250 kHz, `75`=400 kHz, `80`=500 kHz, `100`=840 kHz, `90`=1 MHz.

**PSU voltage configs (Config-350-AllTest.cfg):** `1`=standby, `21`=10 V/1 A,
`22`=20 V, `23`=30 V, ... `37`=170 V/1 A, `38`=180 V/0.5 A, ... `55`=350 V/0.5 A.

⚠ The existing 350 V configs (`38`–`55`) use a 0.5 A current limit. For the high-freq
full-power test the email recommends limiting to **0.3 A**. See the note at the
bottom of the notebook for suggested new PSU configs to add to the NVM.

In [ ]:
# Step 1 — set swB frequency config (switch first, then PSUs).
swB_config = 40   # e.g. 40=1kHz (voltage-ramp phase). Edit and re-run.

with swB.thread_lock:
    status = swB.load_current_config(swB_config)
logger.info(f"swB | load_current_config({swB_config}) -> status={status}")
print(f"swB -> config {swB_config} (status={status})")

In [ ]:
# Step 2 — set PSU2 (Quad1, swB out 0/1) voltage config.
psu2_config = 1   # e.g. 1=standby, 21=10V, 25=50V, 30=100V, ..., 55=350V. Edit & re-run.

with psu2.thread_lock:
    status = psu2.load_current_config(psu2_config)
logger.info(f"psu2 | load_current_config({psu2_config}) -> status={status}")
print(f"psu2 (Quad1) -> config {psu2_config} (status={status})")

In [ ]:
# Step 3 — set PSU1 (IonFunnels, swB out 2/3) voltage config.
psu1_config = 1   # e.g. 1=standby, 21=10V, ..., 55=350V. Edit & re-run.

with psu1.thread_lock:
    status = psu1.load_current_config(psu1_config)
logger.info(f"psu1 | load_current_config({psu1_config}) -> status={status}")
print(f"psu1 (IonFunnels) -> config {psu1_config} (status={status})")

## swA test — PSU3 (Quad2) + PSU4 (Quad3/4)

⚠ **Note:** `config_swA.cfg` currently contains only test DIO / test-clock / static-switch
configurations — **no `SwitchSym` RF configurations yet**. To drive the quadrupoles via swA
you need to add new configurations to the NVM (analogous to swB's 40–90 SwitchSym configs).
See the note at the bottom of the notebook for suggestions.

Until those are added, keep `swA_config = 1` (Standby) to leave swA's HV outputs disabled.

In [ ]:
# Step 1 — set swA frequency config (switch first, then PSUs).
swA_config = 1    # 1=Standby. Set to a SwitchSym config once available in NVM.

with swA.thread_lock:
    status = swA.load_current_config(swA_config)
logger.info(f"swA | load_current_config({swA_config}) -> status={status}")
print(f"swA -> config {swA_config} (status={status})")

In [ ]:
# Step 2 — set PSU3 (Quad2, swA out 0/1) voltage config.
psu3_config = 1   # e.g. 1=standby, 21=10V, ..., 55=350V. Edit & re-run.

with psu3.thread_lock:
    status = psu3.load_current_config(psu3_config)
logger.info(f"psu3 | load_current_config({psu3_config}) -> status={status}")
print(f"psu3 (Quad2) -> config {psu3_config} (status={status})")

In [ ]:
# Step 3 — set PSU4 (Quad3/4, swA out 2/3) voltage config.
psu4_config = 1   # e.g. 1=standby, 21=10V, ..., 55=350V. Edit & re-run.

with psu4.thread_lock:
    status = psu4.load_current_config(psu4_config)
logger.info(f"psu4 | load_current_config({psu4_config}) -> status={status}")
print(f"psu4 (Quad3/4) -> config {psu4_config} (status={status})")

## Emergency shutdown — all devices to standby

Loads standby config (`0`) on every PSU first (HV down), then on both switches.
Run this manually if anything looks wrong. No automatic triggering.

In [ ]:
# PSUs down first, then switches.
STANDBY_CONFIG = 0   # placeholder: standard standby config slot on every device

for name, dev in [("psu1", psu1), ("psu2", psu2), ("psu3", psu3), ("psu4", psu4)]:
    try:
        with dev.thread_lock:
            status = dev.load_current_config(STANDBY_CONFIG)
        logger.warning(f"{name} | SHUTDOWN load_current_config({STANDBY_CONFIG}) -> {status}")
        print(f"{name} -> standby ({status})")
    except Exception as e:
        logger.error(f"{name} shutdown failed: {e}")
        print(f"{name} shutdown FAILED: {e}")

for name, dev in [("swA", swA), ("swB", swB)]:
    try:
        with dev.thread_lock:
            status = dev.load_current_config(STANDBY_CONFIG)
        logger.warning(f"{name} | SHUTDOWN load_current_config({STANDBY_CONFIG}) -> {status}")
        print(f"{name} -> standby ({status})")
    except Exception as e:
        logger.error(f"{name} shutdown failed: {e}")
        print(f"{name} shutdown FAILED: {e}")

print("All devices commanded to standby.")

### Stop Live Plot

In [ ]:
console_handler.setLevel(logging.DEBUG)
logger.info("Live plot logging restored. Bokeh server thread will exit when kernel stops.")
print("Console logging restored. (Browser tab can be closed manually.)")

## Disconnect

In [ ]:
psu1.disconnect()
psu2.disconnect()
psu3.disconnect()
psu4.disconnect()
swA.disconnect()
swB.disconnect()